In [11]:
# 01 - SETUP: imports, configurações e diretórios
import os
from pathlib import Path
import math
import pandas as pd
import numpy as np
from datetime import datetime
import plotly.graph_objects as go
import plotly.express as px

# Paths locais (ajuste se quiser)
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = DATA_DIR / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Exibição pandas
pd.options.display.float_format = "R$ {:,.2f}".format

print("Setup concluído. Base:", BASE_DIR)


Setup concluído. Base: e:\Projetos\sam_analise_financeira\notebooks


In [12]:
# 02 - GLOSSARIO: tenta carregar core/glossary.py, se não, usa fallback mínimo.
try:
    from core.glossary import obter_explicacao  # se você já tiver esse módulo
    print("Usando core.glossary.obter_explicacao()")
except Exception as e:
    print("core.glossary não encontrado — usando glossário fallback.")
    _GLOSS = {
        "MRR": {
            "layman": "MRR = Receita Recorrente Mensal — quanto a empresa recebe por mês de assinaturas.",
            "exec": "MRR: soma das assinaturas ativas no mês; base para projeção de ARR."
        },
        "LTV": {
            "layman": "LTV = Valor que um cliente gera durante toda sua vida útil.",
            "exec": "LTV: ARPU / churn mensal (simplificado)."
        },
        "CAC": {
            "layman": "CAC = quanto custa adquirir um novo cliente (em média).",
            "exec": "CAC: total gasto em aquisição / novos clientes adquiridos."
        },
        "COGS": {
            "layman": "COGS = custos diretamente ligados à entrega do produto (ex.: custo de IA por usuário).",
            "exec": "COGS inclui API LLM, taxas de pagamento e marketing de conversão em algumas linhas."
        },
        "ARPU": {
            "layman": "ARPU = Receita média por usuário.",
            "exec": "ARPU = MRR / clientes ativos."
        },
        "Runway": {
            "layman": "Runway = meses que o caixa atual aguenta antes de zerar (com as despesas atuais).",
            "exec": "Runway = Caixa / (burn mensal médio)."
        }
    }
    def obter_explicacao(chave, publico="layman"):
        return _GLOSS.get(chave, {}).get(publico, "")


core.glossary não encontrado — usando glossário fallback.


In [13]:
# 03 - PREMISSAS (edite os valores aqui conforme quiser)
premissas = {
    "data_inicio": "2025-11-01",
    "meses_projecao": 36,
    "trafego_inicial": 1000,                # visitantes / mês
    "crescimento_trafego_mensal": 0.20,     # 20% mês a mês
    "visitante_para_trial": 0.05,           # 5%
    "trial_para_pago": 0.15,                # 15%
    "usuarios_pagos_iniciais": 50,          # coorte inicial
    "churn_mensal": 0.04,                   # 4% churn
    "preco_lite": 69.90,
    "preco_trader": 99.00,
    "preco_pro": 159.00,
    "mix_lite": 0.50,
    "mix_trader": 0.30,
    "mix_pro": 0.20,
    "custo_ia_por_usuario": 5.00,           # R$ / mês
    "taxa_pagamento_pct": 0.03,             # 3%
    "infra_base_mensal": 709.0,
    "marketing_minimo": 500.0,
    "marketing_reinvest_pct": 0.15,
    "salario_fundador": 10000.0,
    "salario_eng": 8000.0,
    "salario_cs": 5000.0,
    "encargos_pct": 0.68,
    "capex_unico": 8000.0,
    "taxa_impostos_pct": 0.06,
    "caixa_inicial": 2000.0
}

# Exibir premissas resumidas
for k, v in premissas.items():
    print(f"{k}: {v}")


data_inicio: 2025-11-01
meses_projecao: 36
trafego_inicial: 1000
crescimento_trafego_mensal: 0.2
visitante_para_trial: 0.05
trial_para_pago: 0.15
usuarios_pagos_iniciais: 50
churn_mensal: 0.04
preco_lite: 69.9
preco_trader: 99.0
preco_pro: 159.0
mix_lite: 0.5
mix_trader: 0.3
mix_pro: 0.2
custo_ia_por_usuario: 5.0
taxa_pagamento_pct: 0.03
infra_base_mensal: 709.0
marketing_minimo: 500.0
marketing_reinvest_pct: 0.15
salario_fundador: 10000.0
salario_eng: 8000.0
salario_cs: 5000.0
encargos_pct: 0.68
capex_unico: 8000.0
taxa_impostos_pct: 0.06
caixa_inicial: 2000.0


In [14]:
# 04 - FUNCAO: gerar_projecao_financeira(premissas)
def gerar_projecao_financeira(p, meses=None):
    meses = meses or int(p["meses_projecao"])
    start_date = datetime.fromisoformat(p["data_inicio"])
    idx = [ (start_date.replace(day=1) + pd.DateOffset(months=i)).strftime("%Y-%m") for i in range(meses) ]
    
    # arrays
    trafego = np.zeros(meses)
    trials = np.zeros(meses, dtype=int)
    novos_pagantes = np.zeros(meses, dtype=int)
    clientes = np.zeros(meses, dtype=float)
    
    trafego[0] = p["trafego_inicial"]
    for t in range(1, meses):
        trafego[t] = trafego[t-1] * (1 + p["crescimento_trafego_mensal"])
    trials = np.round(trafego * p["visitante_para_trial"]).astype(int)
    novos_pagantes_raw = np.round(trials * p["trial_para_pago"]).astype(int)
    # ramp-up (damping) primeiros 6 meses
    ramp = np.ones(meses)
    ramp[:6] = np.linspace(0.6, 1.0, 6)
    novos_pagantes = (novos_pagantes_raw * ramp).astype(int)
    
    clientes[0] = p["usuarios_pagos_iniciais"]
    for t in range(1, meses):
        clientes[t] = clientes[t-1] * (1 - p["churn_mensal"]) + novos_pagantes[t]
    
    # distribuir por plano
    cust_lite = clientes * p["mix_lite"]
    cust_trader = clientes * p["mix_trader"]
    cust_pro = clientes * p["mix_pro"]
    mrr = cust_lite * p["preco_lite"] + cust_trader * p["preco_trader"] + cust_pro * p["preco_pro"]
    
    # marketing = max(min, reinvest% * prev MRR)
    marketing = np.zeros(meses)
    for t in range(meses):
        if t == 0:
            marketing[t] = p["marketing_minimo"]
        else:
            marketing[t] = max(p["marketing_minimo"], mrr[t-1] * p["marketing_reinvest_pct"])
    
    # folha (simplificada): fundador sempre, eng começa no mês 12, cs no mês 18
    folha = np.zeros(meses)
    folha += p["salario_fundador"] * (1 + p["encargos_pct"])
    for t in range(11, meses):  # mês 12 índice 11
        folha[t] += p["salario_eng"] * (1 + p["encargos_pct"])
    for t in range(17, meses):  # mês 18 índice 17
        folha[t] += p["salario_cs"] * (1 + p["encargos_pct"])
    
    # custos variáveis
    custo_ia = clientes * p["custo_ia_por_usuario"]
    taxas_pag = mrr * p["taxa_pagamento_pct"]
    cogs = custo_ia + taxas_pag + marketing  # incluí marketing como custo de crescimento (pode ajustar)
    
    # opex fixo
    infra = np.full(meses, p["infra_base_mensal"])
    opex_total = folha + infra
    
    # financeiro
    receita = mrr.copy()
    lucro_bruto = receita - (custo_ia + taxas_pag)
    ebitda = lucro_bruto - (folha + infra) + marketing*0  # se quiser não duplicar marketing, remova +marketing acima
    depreciacao = np.zeros(meses); depreciacao[0] = 0; # usaremos CAPEX como desembolso inicial
    depreciacao = np.full(meses, (p["capex_unico"] / 60.0))  # linear 5 anos
    impostos = np.where(ebitda > 0, ebitda * p["taxa_impostos_pct"], 0.0)
    lucro_liquido = ebitda - depreciacao - impostos
    
    # caixa
    caixa = np.zeros(meses)
    caixa[0] = p["caixa_inicial"] + lucro_liquido[0] - p["capex_unico"]
    for t in range(1, meses):
        caixa[t] = caixa[t-1] + lucro_liquido[t]  # capex só no mês 1
    
    # KPIs
    arpu = np.where(clientes>0, mrr / clientes, 0.0)
    ltv = np.where(p["churn_mensal"]>0, arpu / p["churn_mensal"], 0.0)
    # CAC rolling 12: marketing last 12 / novos_pagantes last 12
    cac = np.full(meses, np.nan)
    for t in range(meses):
        start = max(0, t-11)
        new12 = novos_pagantes[start:t+1].sum()
        mar12 = marketing[start:t+1].sum()
        cac[t] = (mar12 / new12) if new12>0 else np.nan
    
    df = pd.DataFrame({
        "mes": idx,
        "trafego": trafego.astype(int),
        "trials": trials,
        "novos_pagantes": novos_pagantes,
        "clientes_ativos": np.round(clientes).astype(int),
        "cust_lite": np.round(cust_lite).astype(int),
        "cust_trader": np.round(cust_trader).astype(int),
        "cust_pro": np.round(cust_pro).astype(int),
        "mrr": mrr.round(2),
        "marketing": marketing.round(2),
        "custo_ia": custo_ia.round(2),
        "taxas_pag": taxas_pag.round(2),
        "cogs": cogs.round(2),
        "folha": folha.round(2),
        "infra": infra.round(2),
        "opex_total": opex_total.round(2),
        "lucro_bruto": lucro_bruto.round(2),
        "ebitda": ebitda.round(2),
        "depreciacao": depreciacao.round(2),
        "impostos": impostos.round(2),
        "lucro_liquido": lucro_liquido.round(2),
        "caixa": caixa.round(2),
        "arpu": np.round(arpu,2),
        "ltv": np.round(ltv,2),
        "cac": np.round(cac,2)
    })
    return df

# Gerar DataFrame exemplo (usa premissas padrão)
df_base = gerar_projecao_financeira(premissas, meses=premissas["meses_projecao"])
df_base.head(8)


,mes,trafego,trials,novos_pagantes,clientes_ativos,cust_lite,cust_trader,cust_pro,mrr,marketing,...,opex_total,lucro_bruto,ebitda,depreciacao,impostos,lucro_liquido,caixa,arpu,ltv,cac
0,2025-11,1000,50,4,50,25,15,10,"R$ 4,822.50",R$ 500.00,...,"R$ 17,509.00","R$ 4,427.82","R$ -13,081.18",R$ 133.33,R$ 0.00,"R$ -13,214.51","R$ -19,214.51",R$ 96.45,"R$ 2,411.25",R$ 125.00
1,2025-12,1200,60,6,54,27,16,11,"R$ 5,208.30",R$ 723.38,...,"R$ 17,509.00","R$ 4,782.05","R$ -12,726.95",R$ 133.33,R$ 0.00,"R$ -12,860.28","R$ -32,074.79",R$ 96.45,"R$ 2,411.25",R$ 122.34
2,2026-01,1440,72,8,60,30,18,12,"R$ 5,771.57",R$ 781.24,...,"R$ 17,509.00","R$ 5,299.22","R$ -12,209.78",R$ 133.33,R$ 0.00,"R$ -12,343.11","R$ -44,417.90",R$ 96.45,"R$ 2,411.25",R$ 111.37
3,2026-02,1728,86,10,67,34,20,13,"R$ 6,505.21",R$ 865.74,...,"R$ 17,509.00","R$ 5,972.82","R$ -11,536.18",R$ 133.33,R$ 0.00,"R$ -11,669.52","R$ -56,087.42",R$ 96.45,"R$ 2,411.25",R$ 102.51
4,2026-03,2073,104,14,79,39,24,16,"R$ 7,595.30",R$ 975.78,...,"R$ 17,509.00","R$ 6,973.70","R$ -10,535.30",R$ 133.33,R$ 0.00,"R$ -10,668.64","R$ -66,756.06",R$ 96.45,"R$ 2,411.25",R$ 91.57
5,2026-04,2488,124,19,95,47,28,19,"R$ 9,124.04","R$ 1,139.29",...,"R$ 17,509.00","R$ 8,377.32","R$ -9,131.68",R$ 133.33,R$ 0.00,"R$ -9,265.01","R$ -76,021.07",R$ 96.45,"R$ 2,411.25",R$ 81.73
6,2026-05,2985,149,22,113,56,34,23,"R$ 10,880.97","R$ 1,368.61",...,"R$ 17,509.00","R$ 9,990.47","R$ -7,518.53",R$ 133.33,R$ 0.00,"R$ -7,651.86","R$ -83,672.93",R$ 96.45,"R$ 2,411.25",R$ 76.55
7,2026-06,3583,179,27,135,68,41,27,"R$ 13,049.88","R$ 1,632.15",...,"R$ 17,509.00","R$ 11,981.88","R$ -5,527.12",R$ 133.33,R$ 0.00,"R$ -5,660.46","R$ -89,333.39",R$ 96.45,"R$ 2,411.25",R$ 72.60


In [15]:
# 05 - KPIS RESUMO e pontos críticos (vale da morte, break-even)
df = df_base.copy()

# ponto de menor caixa
indice_min_caixa = df["caixa"].idxmin()
mes_min_caixa = df.loc[indice_min_caixa, "mes"]
valor_min_caixa = df.loc[indice_min_caixa, "caixa"]

# break-even (primeiro mês com EBITDA > 0)
be_idx = df.index[df["ebitda"] > 0]
mes_break_even = df.loc[be_idx[0],"mes"] if len(be_idx)>0 else None

# runway aproximado = meses até caixa < 0 (se cair) ou caixa/avg burn
burn_medio = df.loc[:, "opex_total"].mean()
runway_meses = df.loc[0,"caixa"] / (burn_medio if burn_medio>0 else 1)

summary = {
    "MRR mes 1": df.loc[0,"mrr"],
    "MRR mes 12": df.loc[min(11, len(df)-1),"mrr"],
    "MRR mes 36": df.loc[len(df)-1,"mrr"],
    "Menor caixa (mes)": mes_min_caixa,
    "Menor caixa (valor)": valor_min_caixa,
    "Break-even mes (primeiro EBITDA>0)": mes_break_even,
    "Runway aproximado (meses)": round(runway_meses,1)
}

pd.Series(summary)


MRR mes 1                                R$ 4,822.50
MRR mes 12                              R$ 26,941.60
MRR mes 36                           R$ 2,136,461.59
Menor caixa (mes)                            2026-11
Menor caixa (valor)                    R$ -98,038.75
Break-even mes (primeiro EBITDA>0)           2026-09
Runway aproximado (meses)                   R$ -0.60
dtype: object

In [16]:
# 06 - GRAFICO INTERATIVO: MRR vs Caixa com legenda explicativa (usando glossary)
x = df["mes"]
fig = go.Figure()
fig.add_trace(go.Scatter(x=x, y=df["mrr"], name="MRR (Receita Recorrente Mensal)", mode="lines+markers"))
fig.add_trace(go.Scatter(x=x, y=df["caixa"], name="Caixa (Saldo Acumulado)", mode="lines+markers", yaxis="y2"))

fig.update_layout(
    title="Projeção: MRR vs Caixa",
    xaxis_title="Mês",
    yaxis=dict(title="MRR (R$)"),
    yaxis2=dict(title="Caixa (R$)", overlaying="y", side="right"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

# Adicionar explicações resumidas (legenda textual abaixo do gráfico)
exp_mrr = obter_explicacao("MRR", "layman")
exp_caixa = "Caixa: saldo acumulado mês a mês (considera lucros e CAPEX inicial)."
nota = f"MRR: {exp_mrr}<br>Caixa: {exp_caixa}"

fig.add_annotation(
    text=nota,
    xref="paper", yref="paper", x=0, y=-0.25,
    showarrow=False, align="left"
)

fig.update_layout(margin=dict(b=140))
fig.show()


In [17]:
# 07 - TORANDO: função para sensibilidade univariada e plot tornado
def tornado_analysis(base_p, var_name, values_list):
    base_df = gerar_projecao_financeira(base_p, meses=base_p["meses_projecao"])
    base_cash = base_df.loc[base_df.index[-1], "caixa"]
    results = []
    for v in values_list:
        p = base_p.copy()
        p[var_name] = v
        df = gerar_projecao_financeira(p, meses=p["meses_projecao"])
        results.append((v, df.loc[df.index[-1], "caixa"]))
    return base_cash, results

# Variáveis para teste
vars_to_test = {
    "churn_mensal": [0.02, 0.03, 0.04, 0.05, 0.06],
    "crescimento_trafego_mensal": [0.10, 0.15, 0.20, 0.25, 0.30],
    "custo_ia_por_usuario": [2.0, 3.5, 5.0, 7.5, 10.0]
}

# Executar tornado para cada variável
tornado_results = {}
for var, vals in vars_to_test.items():
    base_cash, res = tornado_analysis(premissas, var, vals)
    tornado_results[var] = {"base": base_cash, "res": res}

# Plot tornado simplificado (valores absolutos vs baseline)
rows = []
for var, data in tornado_results.items():
    base = data["base"]
    for v, cash in data["res"]:
        rows.append({"variavel": var, "valor_testado": v, "caixa_final": cash, "delta": cash - base})
tornado_df = pd.DataFrame(rows)
# Para visual: agregue extremos por variavel
fig = px.bar(tornado_df, x="delta", y="variavel", color="valor_testado", orientation="h",
             title="Tornado: impacto das variáveis no Caixa final (Mês 36)")
fig.update_layout(barmode="group")
fig.show()


In [18]:
# 08 - EXPORT: salvar CSVs para consumo do app / streamlit
ts = datetime.now().strftime("%Y%m%d_%H%M%S")
csv_proj = OUTPUT_DIR / f"projecao_sam_{ts}.csv"
csv_kpis = OUTPUT_DIR / f"kpis_sam_{ts}.csv"

df.to_csv(csv_proj, index=False)
pd.Series(summary).to_csv(csv_kpis, header=False)

print("Arquivos salvos em:", OUTPUT_DIR)
print("-", csv_proj.name)
print("-", csv_kpis.name)


Arquivos salvos em: e:\Projetos\sam_analise_financeira\notebooks\data\outputs
- projecao_sam_20251113_174104.csv
- kpis_sam_20251113_174104.csv


In [19]:
# 09 - HELPER: atualizar premissas rapidamente e re-executar (útil para Streamlit)
def atualizar_premissas_novo(d_updates):
    """
    d_updates = dict com chaves de premissas a trocar, ex: {"churn_mensal": 0.05}
    Retorna novo df e summary
    """
    p = premissas.copy()
    p.update(d_updates)
    df_new = gerar_projecao_financeira(p, meses=p["meses_projecao"])
    # resumo rápido
    idx_min = df_new["caixa"].idxmin()
    resumo = {
        "menor_caixa_mes": df_new.loc[idx_min, "mes"],
        "menor_caixa_valor": df_new.loc[idx_min,"caixa"],
        "mrr_mes12": df_new.loc[min(11,len(df_new)-1),"mrr"],
        "mrr_mes36": df_new.loc[len(df_new)-1,"mrr"]
    }
    return df_new, resumo

# Exemplo de uso:
# novo_df, novo_resumo = atualizar_premissas_novo({"churn_mensal": 0.05})
# novo_df.head()


# Conclusão - Notebook 01 (Base de Modelagem)

Este notebook gera **toda** a projeção financeira do SAM a partir de premissas configuráveis (nenhuma planilha externa necessária).  
O fluxo é:

1. Definir/editar as premissas na célula 03.
2. Executar a célula 04 para gerar a projeção (DataFrame completo).
3. Consultar KPIs na célula 05 e visualizar o gráfico interativo na célula 06.
4. Fazer análises de sensibilidade com a célula 07.
5. Exportar dados com a célula 08 para uso pelo app (Streamlit) ou para análise adicional.

### Recomendações imediatas
- Integre este notebook ao Streamlit chamando `gerar_projecao_financeira()` no backend (ou via FastAPI) e exponha as premissas como sliders/inputs.
- Use `atualizar_premissas_novo()` como endpoint para recalcular rápido com novos valores.
- Mantenha o glossário atualizado (core/glossary.py) para alimentar as legendas e explicações automáticas nos gráficos.

Se quiser, gero agora os **cadernos 02–05** completos (código funcional, não pseudo) no mesmo formato — ou **faço ajustes** no notebook 01 conforme os seus números reais / cenários preferidos. Quer que eu já gere os notebooks 02 e 03 prontos para colar também? 
